<a href="https://colab.research.google.com/github/Sai-nikhil2k5/Cardio-Pulmonary-disease-detection-by-sounds/blob/main/Cardio-Pulmonary_Disease%20detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

os.listdir('/content/drive/MyDrive')

['GDToT',
 'AppDrive',
 '1086289 _Twitter accounts.xlsx',
 'yaswanthyashuuuu-20240111-0002.jpg',
 'IMG_20240128_191522.jpg',
 'A001C00172_240310_7MLF~3.mp4',
 'Screenshot_20240311_135630.jpg',
 'AVL TREES.pdf',
 'Classroom',
 'Screenshot_20240910_170743 (1).jpg',
 'Screenshot_20240910_170743.jpg',
 'Path finder-WPS Office.docx',
 'IMG-20231103-WA0009.jpg',
 'IMG_20240909_211011.jpg',
 'IMG_20241019_200053.jpg',
 'Colab Notebooks',
 'train',
 'valid',
 'VWCC',
 'Screenshot (32).png',
 'Screenshot (34).png',
 'Sai nikhil resume.pdf',
 'New PDF Document-WPS Office (1).pdf',
 'New PDF Document-WPS Office.pdf',
 'OfferLetter.pdf',
 'Screenshot_2025-05-11-22-03-58-13_6012fa4d4ddec268fc5c7112cbb265e7.jpg',
 'Untitled document.gdoc',
 'Screenshot_2025-05-26-12-54-00-33_1c337646f29875672b5a61192b9010f9.jpg',
 'Screenshot_2025-05-26-12-54-13-74_af7491585f6f34cc867e22eb0718f0bc.jpg',
 'Vizag_GreenMask.tif',
 'VizagGreenVisualized.tif',
 'Screenshot_2025-06-07-20-39-12-01_944a2809ea1b4cda6ef12d1db

In [4]:
import zipfile

zip_path = "/content/drive/MyDrive/Dataset.v2.zip"
extract_path = "/content/Dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Extracted!")

Extracted!


In [8]:
DATA_PATH = "/content/Dataset/Dataset.v2"

print(os.listdir(DATA_PATH))

['LS', 'LS.csv', 'Mix2', 'Mix1', 'Mix3', 'HS.csv', 'Mix.csv', 'HS']


In [22]:
import pandas as pd

DATA_PATH = "/content/Dataset/Dataset.v2"

df_hs = pd.read_csv(DATA_PATH + "/HS.csv")
df_ls = pd.read_csv(DATA_PATH + "/LS.csv")
df_mix = pd.read_csv(DATA_PATH + "/Mix.csv")

In [26]:
df_hs_new = pd.DataFrame({
    "file": df_hs["Heart Sound ID"],
    "heart_label": df_hs["Heart Sound Type"],
    "lung_label": "None",
    "source": "heart"
})

df_ls_new = pd.DataFrame({
    "file": df_ls["Lung Sound ID"],
    "heart_label": "None",
    "lung_label": df_ls["Lung Sound Type"],
    "source": "lung"
})

df_mix3 = pd.DataFrame({
    "file": df_mix["Mixed Sound ID"],
    "heart_label": df_mix["Heart Sound Type"],
    "lung_label": df_mix["Lung Sound Type"],
    "source": "mix3"
})

df_mix1 = pd.DataFrame({
    "file": df_mix["Heart Sound ID"],
    "heart_label": df_mix["Heart Sound Type"],
    "lung_label": "None",
    "source": "mix1"
})

df_mix2 = pd.DataFrame({
    "file": df_mix["Lung Sound ID"],
    "heart_label": "None",
    "lung_label": df_mix["Lung Sound Type"],
    "source": "mix2"
})

df = pd.concat([
    df_hs_new,
    df_ls_new,
    df_mix1,
    df_mix2,
    df_mix3
], ignore_index=True)

print("Total samples:", len(df))


Total samples: 535


,file,heart_label,lung_label,source
0,F_N_RC,Normal,None,heart
1,F_N_LC,Normal,None,heart
2,M_N_RUSB,Normal,None,heart
3,F_N_LUSB,Normal,None,heart
4,F_N_LLSB,Normal,None,heart


In [27]:
df.tail()

,file,heart_label,lung_label,source
530,M0141,S4,Wheezing,mix3
531,M0142,Mid Systolic Murmur,Normal,mix3
532,M0143,Early Systolic Murmur,Wheezing,mix3
533,M0144,AV Block,Normal,mix3
534,M0145,Late Diastolic Murmur,Normal,mix3


In [28]:
def get_path(row):
    if row["source"] == "heart":
        return f"{DATA_PATH}/HS/HS/{row['file']}.wav"

    elif row["source"] == "lung":
        return f"{DATA_PATH}/LS/LS/{row['file']}.wav"

    elif row["source"] == "mix1":
        return f"{DATA_PATH}/Mix1/{row['file']}.wav"

    elif row["source"] == "mix2":
        return f"{DATA_PATH}/Mix2/{row['file']}.wav"

    else:  # mix3
        return f"{DATA_PATH}/Mix3/{row['file']}.wav"

df["path"] = df.apply(get_path, axis=1)

# check
print(df["path"].head())

0      /content/Dataset/Dataset.v2/HS/HS/F_N_RC.wav
1      /content/Dataset/Dataset.v2/HS/HS/F_N_LC.wav
2    /content/Dataset/Dataset.v2/HS/HS/M_N_RUSB.wav
3    /content/Dataset/Dataset.v2/HS/HS/F_N_LUSB.wav
4    /content/Dataset/Dataset.v2/HS/HS/F_N_LLSB.wav
Name: path, dtype: object


In [31]:
import os

print(os.path.exists(df.iloc[12]["path"]))

True


In [32]:
from sklearn.preprocessing import LabelEncoder

heart_encoder = LabelEncoder()
lung_encoder = LabelEncoder()

df["heart_encoded"] = heart_encoder.fit_transform(df["heart_label"])
df["lung_encoded"] = lung_encoder.fit_transform(df["lung_label"])

print(df[["heart_label", "heart_encoded"]].drop_duplicates())
print(df[["lung_label", "lung_encoded"]].drop_duplicates())

              heart_label  heart_encoded
0                  Normal              7
6   Late Diastolic Murmur              3
7     Mid Systolic Murmur              5
11   Late Systolic Murmur              4
13    Atrial Fibrillation              1
14                     S4              9
15  Early Systolic Murmur              2
22                     S3              8
28            Tachycardia             10
33               AV Block              0
50                   None              6
         lung_label  lung_encoded
0              None             2
50           Normal             3
56      Pleural Rub             4
58          Rhonchi             5
59         Wheezing             6
60    Fine Crackles             1
62  Coarse Crackles             0


In [41]:
import librosa
import numpy as np

def extract_features(file_path):
    try:
        audio, sr = librosa.load(file_path, sr=22050)
        mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
        return np.mean(mfcc.T, axis=0)
    except:
        return None

In [43]:
file_path = df.iloc[0]["path"]
audio, sr = librosa.load(file_path, sr=22050)
mfcc = librosa.feature.mfcc(y=audio, sr=sr, n_mfcc=13)
print(mfcc.shape)

(13, 646)


In [44]:
features = extract_features(file_path)
print(features.shape)

(13,)


In [53]:
df.shape

(535, 7)

In [55]:
failed_files = []

X = []
y_heart = []
y_lung = []

for _, row in df.iterrows():
    features = extract_features(row["path"])

    if features is not None:
        X.append(features)
        y_heart.append(row["heart_encoded"])
        y_lung.append(row["lung_encoded"])
    else:
        failed_files.append(row["path"])

print("Failed files:", len(failed_files))

/tmp/ipykernel_15407/3212857617.py:6: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(file_path, sr=22050)
/usr/local/lib/python3.12/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


Failed files: 15


In [56]:
print(failed_files[:10])

['/content/Dataset/Dataset.v2/LS/LS/M_C_LUA.wav', '/content/Dataset/Dataset.v2/LS/LS/M_G_LLA.wav', '/content/Dataset/Dataset.v2/LS/LS/M_G_LUA.wav', '/content/Dataset/Dataset.v2/LS/LS/F_C_LUA.wav', '/content/Dataset/Dataset.v2/LS/LS/M_G_RLA.wav', '/content/Dataset/Dataset.v2/LS/LS/M_C_RUA.wav', '/content/Dataset/Dataset.v2/LS/LS/F_G_LMA.wav', '/content/Dataset/Dataset.v2/LS/LS/F_G_RLA.wav', '/content/Dataset/Dataset.v2/LS/LS/M_C_RLA.wav', '/content/Dataset/Dataset.v2/LS/LS/F_G_RMA.wav']


In [57]:
X = np.array(X)
y_heart = np.array(y_heart)
y_lung = np.array(y_lung)

print(X.shape)
print(y_heart.shape)
print(y_lung.shape)

(520, 13)
(520,)
(520,)


In [58]:
print(len(X), len(y_heart), len(y_lung))

520 520 520


In [59]:
from sklearn.model_selection import train_test_split

X_train, X_test, yh_train, yh_test, yl_train, yl_test = train_test_split(
    X, y_heart, y_lung, test_size=0.2, random_state=42
)

print("Train:", X_train.shape)
print("Test:", X_test.shape)

Train: (416, 13)
Test: (104, 13)


In [60]:
from sklearn.ensemble import RandomForestClassifier

heart_model = RandomForestClassifier(n_estimators=200)
heart_model.fit(X_train, yh_train)

RandomForestClassifier(n_estimators=200)

In [61]:
lung_model = RandomForestClassifier(n_estimators=200)
lung_model.fit(X_train, yl_train)

RandomForestClassifier(n_estimators=200)

In [62]:
yh_pred = heart_model.predict(X_test)
yl_pred = lung_model.predict(X_test)

In [63]:
from sklearn.metrics import accuracy_score, classification_report

print("Heart Accuracy:", accuracy_score(yh_test, yh_pred))
print("Lung Accuracy:", accuracy_score(yl_test, yl_pred))

print("\nHeart Report:\n", classification_report(yh_test, yh_pred))
print("\nLung Report:\n", classification_report(yl_test, yl_pred))

Heart Accuracy: 0.6442307692307693
Lung Accuracy: 0.7788461538461539

Heart Report:
               precision    recall  f1-score   support

           0       0.60      0.86      0.71         7
           1       0.17      0.50      0.25         2
           2       0.60      0.75      0.67         4
           3       1.00      0.60      0.75         5
           4       0.78      0.88      0.82         8
           5       0.67      0.25      0.36         8
           6       0.73      0.93      0.81        40
           7       1.00      0.25      0.40         8
           8       1.00      0.10      0.18        10
           9       0.38      0.43      0.40         7
          10       0.33      0.40      0.36         5

    accuracy                           0.64       104
   macro avg       0.66      0.54      0.52       104
weighted avg       0.72      0.64      0.61       104


Lung Report:
               precision    recall  f1-score   support

           0       0.00      0.0

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [64]:
sample = X_test[0].reshape(1, -1)

heart_pred = heart_model.predict(sample)
lung_pred = lung_model.predict(sample)

print("Heart:", heart_encoder.inverse_transform(heart_pred))
print("Lung:", lung_encoder.inverse_transform(lung_pred))

Heart: ['None']
Lung: ['Normal']
